# Palmer Indices With xarray Inputs

The Palmer implementation is currently a NumPy API. This notebook shows the v2.5 bridge pattern: keep labeled xarray inputs at workflow boundaries, call `palmer.pdsi()` on values, then rewrap the outputs with coordinates and metadata.

In [1]:
import logging

import numpy as np
import pandas as pd
import xarray as xr

from climate_indices import palmer

logging.disable(logging.CRITICAL)

In [2]:
time = pd.date_range("2000-01-01", periods=10 * 12, freq="MS")
month_index = np.arange(time.size)

precip_in = xr.DataArray(
    3.0 + 0.8 * np.sin(2.0 * np.pi * month_index / 12.0) + 0.1 * np.sin(month_index),
    coords={"time": time},
    dims="time",
    name="precipitation",
    attrs={"units": "inches"},
)
pet_in = xr.DataArray(
    2.0 + 0.5 * np.cos(2.0 * np.pi * month_index / 12.0),
    coords={"time": time},
    dims="time",
    name="pet",
    attrs={"units": "inches"},
)

pdsi, phdi, pmdi, zindex, params = palmer.pdsi(
    precip_in.values,
    pet_in.values,
    awc=4.5,
    data_start_year=2000,
    calibration_year_initial=2000,
    calibration_year_final=2009,
)

In [3]:
palmer_ds = xr.Dataset(
    data_vars={
        "pdsi": ("time", pdsi, {"long_name": "Palmer Drought Severity Index", "units": "dimensionless"}),
        "phdi": ("time", phdi, {"long_name": "Palmer Hydrological Drought Index", "units": "dimensionless"}),
        "pmdi": ("time", pmdi, {"long_name": "Palmer Modified Drought Index", "units": "dimensionless"}),
        "zindex": ("time", zindex, {"long_name": "Palmer Z-Index", "units": "dimensionless"}),
    },
    coords={"time": time},
    attrs={"awc_inches": 4.5, "calibration_period": "2000-2009"},
)

assert set(palmer_ds.data_vars) == {"pdsi", "phdi", "pmdi", "zindex"}
assert palmer_ds.sizes["time"] == time.size
assert params is not None

palmer_ds

<xarray.Dataset> Size: 5kB
Dimensions:  (time: 120)
Coordinates:
  * time     (time) datetime64[us] 960B 2000-01-01 2000-02-01 ... 2009-12-01
Data variables:
    pdsi     (time) float64 960B 0.04737 0.7965 1.465 ... -0.5418 -1.338 -1.824
    phdi     (time) float64 960B 0.04737 0.7965 1.465 ... 0.8013 -1.338 -1.824
    pmdi     (time) float64 960B 0.04737 0.7965 1.465 ... -0.1415 -1.338 -1.824
    zindex   (time) float64 960B 0.1421 2.262 2.253 ... -1.625 -2.555 -1.871
Attributes:
    awc_inches:          4.5
    calibration_period:  2000-2009